# Hito 1 — Inspección Inicial y Calidad de Datos

**Proyecto Integrador — Minería de Datos 1**

**Dataset:** `streaming_users_dirty.json` (usuarios de una plataforma de streaming)

## Objetivo de este notebook
Antes de limpiar o analizar los datos, necesitamos entender **qué tenemos**: cuántos registros hay, qué columnas existen, y sobre todo, **qué problemas de calidad** presenta el dataset. Esta inspección es la base que va a justificar las decisiones de limpieza que tomemos en el Hito 2.

No vamos a corregir nada todavía. Solo vamos a **observar y documentar**.


## 1. Carga de datos

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_json("streaming_users_dirty.json")
df.head()


## 2. Estructura general del dataset

Vemos cuántas filas y columnas tiene, qué tipo de dato tiene cada columna, y un resumen estadístico rápido.

In [ ]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.dtypes


In [ ]:
df.describe(include="all")


**Columnas del dataset:**

| Columna | Descripción |
|---|---|
| `user_id` | Identificador único del usuario |
| `age` | Edad del usuario |
| `subscription_plan` | Plan de suscripción (Básico, Estándar, Premium) |
| `monthly_watch_time_mins` | Minutos de consumo mensual |
| `country` | País del usuario |
| `favorite_genre` | Género favorito |
| `last_login_date` | Fecha del último ingreso |
| `customer_support_tickets` | Cantidad de tickets de soporte generados |


## 3. Valores nulos

In [ ]:
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(2)
pd.DataFrame({"nulos": nulos, "% del total": porcentaje})


**Hallazgo:** `favorite_genre` y `last_login_date` presentan valores nulos. Las demás columnas están completas. Esto ya nos anticipa que en el Hito 2 vamos a tener que decidir cómo tratar estos faltantes (imputar, dejar como categoría aparte, etc.).

## 4. Registros duplicados

In [ ]:
# Duplicados exactos (toda la fila igual)
duplicados_totales = df.duplicated().sum()
print("Filas totalmente duplicadas:", duplicados_totales)

# Duplicados por user_id (el identificador no debería repetirse)
duplicados_id = df["user_id"].duplicated().sum()
print("user_id repetidos:", duplicados_id)


**Hallazgo:** Hay filas completamente duplicadas y además `user_id` se repite más veces de las que se explican solo por esas filas duplicadas. Esto sugiere que también existen usuarios cargados más de una vez con datos distintos (posible error de carga). Es un punto a resolver en el Hito 2.

## 5. Inconsistencias en variables categóricas

In [ ]:
df["subscription_plan"].value_counts()


In [ ]:
df["country"].value_counts()


In [ ]:
df["favorite_genre"].value_counts(dropna=False)


**Hallazgo:** Las tres columnas categóricas tienen el mismo problema: **la misma categoría está escrita de distintas formas**.

- `subscription_plan`: por ejemplo "Básico", "basico", "BASICO", "Basic" y "Std" parecen representar el mismo plan.
- `country`: por ejemplo "Colombia", "colombia" y "COL" parecen ser el mismo país.
- `favorite_genre`: por ejemplo "Comedia", "COMEDIA", "comedy" y "Comedia " (con espacio) parecen ser el mismo género.

Esto es típico de un dataset con errores de tipeo, uso de mayúsculas/minúsculas mezcladas, abreviaturas y espacios extra. Se deberá estandarizar en el Hito 2.

## 6. Valores fuera de rango (posibles outliers o errores)

In [ ]:
print("Edad — mínimo:", df["age"].min(), " máximo:", df["age"].max())
print("Minutos de consumo — mínimo:", df["monthly_watch_time_mins"].min(), " máximo:", df["monthly_watch_time_mins"].max())
print("Tickets de soporte — mínimo:", df["customer_support_tickets"].min(), " máximo:", df["customer_support_tickets"].max())


In [ ]:
# Cantidad de registros con valores claramente imposibles
print("Edades negativas o mayores a 100:", ((df["age"] < 0) | (df["age"] > 100)).sum())
print("Minutos de consumo negativos:", (df["monthly_watch_time_mins"] < 0).sum())
print("Tickets de soporte negativos:", (df["customer_support_tickets"] < 0).sum())


**Hallazgo:** Encontramos valores que no tienen sentido en el mundo real:
- Edades negativas y edades de 150 años.
- Minutos de consumo negativos y un valor extremo de 99999 minutos (~69 días de consumo en un mes, imposible).
- Tickets de soporte negativos.

Estos son errores de carga o de sensor, no representan usuarios reales. Se deberán tratar en el Hito 2 (ya sea eliminando esos registros o corrigiéndolos si hay evidencia suficiente).

## 7. Formato de fechas

In [ ]:
import re

fechas = df["last_login_date"].dropna()

formato_estandar = fechas.str.match(r"^\d{4}-\d{2}-\d{2}$")
print("Fechas en formato AAAA-MM-DD:", formato_estandar.sum())
print("Fechas en otro formato:", (~formato_estandar).sum())

fechas[~formato_estandar].head(10)


**Hallazgo:** La mayoría de las fechas siguen el formato `AAAA-MM-DD`, pero hay un grupo importante en otros formatos (`MM-DD-AAAA`, `AAAA/MM/DD`). Antes de poder trabajar con estas fechas como tipo fecha real, hay que unificar el formato.

## 8. Resumen de hallazgos — Calidad inicial del dataset

| Problema detectado | Columna(s) afectada(s) | Impacto |
|---|---|---|
| Valores nulos | `favorite_genre`, `last_login_date` | Requiere decisión de imputación o categoría "sin dato" |
| Filas y `user_id` duplicados | Todo el dataset | Puede inflar el análisis si no se eliminan |
| Categorías escritas de forma inconsistente | `subscription_plan`, `country`, `favorite_genre` | Impide agrupar correctamente sin estandarizar |
| Valores fuera de rango / imposibles | `age`, `monthly_watch_time_mins`, `customer_support_tickets` | Distorsionan estadísticas y análisis si no se tratan |
| Formatos de fecha mixtos | `last_login_date` | Impide convertir la columna a tipo fecha directamente |

Estas observaciones, basadas en evidencia concreta del dataset, son las que van a justificar cada decisión de limpieza que tomemos en el **Hito 2 (Preparación de datos)**. No se corrige nada en este notebook: el objetivo acá era únicamente **documentar el estado real del dataset**.
